In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50
from tqdm.auto import tqdm

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [16]:
DATASET_ROOT = Path("/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/02-Databases/Mammo-Bench/c86fb00c-0fb8-4e0e-85a2-4d415f9c1ada_1a9410d8-9769-4064-a064-0160f2fd193d_DATASET-FILE_Mammo_Bench_zip_20241225112148174/Mammo_Data/Mammo-Bench")

class CSVDataset(Dataset):
    def __init__(self, csv_file, split, root_dir="", transform=None, label_to_idx=None):
        self.data = pd.read_csv(csv_file)
        self.data = self.data[self.data["split"] == split].reset_index(drop=True)

        self.root_dir = Path(root_dir)
        self.transform = transform

        if label_to_idx is None:
            labels = sorted(self.data["classification"].unique())
            self.label_to_idx = {label: i for i, label in enumerate(labels)}
        else:
            self.label_to_idx = label_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        img_path = Path(self.root_dir) / Path(row["preprocessed_image_path"])
        image = Image.open(img_path).convert("RGB")

        label = self.label_to_idx[row["classification"]]

        if self.transform:
            image = self.transform(image)

        return image, label

In [17]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    #transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [18]:
MANIFEST_PATH = "/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/13-PregradoJulian/Federal Learning/infraestructura federada/Federal-Learning/manifests/mamo-bench-split-no-ddsm-rsna.csv"

train_processed = CSVDaimage = (image-127.5)*2 / 255taset(csv_file=MANIFEST_PATH, split="train", root_dir=DATASET_ROOT, transform=train_transform)
# val/test reuse train's label_to_idx so "Benign"/"Malignant" map to the same
# class index in every split (previously each split computed its own mapping
# independently from its local sorted unique labels).
validation_processed = CSVDataset(csv_file=MANIFEST_PATH, split="val", root_dir=DATASET_ROOT, transform=val_transform, label_to_idx=train_processed.label_to_idx)
test_processed = CSVDataset(csv_file=MANIFEST_PATH, split="test", root_dir=DATASET_ROOT, transform=val_transform, label_to_idx=train_processed.label_to_idx)

SyntaxError: invalid decimal literal (1566023376.py, line 3)

In [ ]:
train = DataLoader(train_processed, batch_size=64, shuffle=True)
validation = DataLoader(validation_processed, batch_size=64, shuffle=False)
test = DataLoader(test_processed, batch_size=64, shuffle=False)

In [ ]:
class Classifier(nn.Module):
    def __init__(self, num_class):
        super().__init__()
        self.drop_out = nn.Dropout()
        self.linear = nn.Linear(2048, num_class)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.drop_out(x)
        x = self.linear(x)
        #x = torch.softmax(x, dim=-1)
        return x


class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        base_model = resnet50(weights=None)
        encoder_layers = list(base_model.children())
        self.backbone = nn.Sequential(*encoder_layers[:9])
                        
    def forward(self, x):
        return self.backbone(x)

In [ ]:
# backbone is created below from RadImageNet weights (load_radimagenet_backbone);
# only the classifier head needs its own fresh init here.
classifier = Classifier(num_class=2).to(device)

In [ ]:
# Example: load RadImageNet weights into a ResNet50 backbone
# Replace this path with the actual .pth or .pt file you downloaded.
radimagenet_weights_path = "/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/13-PregradoJulian/Federal Learning/infraestructura federada/Federal-Learning/weights/RadImageNet-resnet50.pth"


def load_radimagenet_backbone(weights_path, device="cpu"):
    model = resnet50(weights=None)
    checkpoint = torch.load(weights_path, map_location=device)

    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    # Remove common prefixes from checkpoints
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

    # The official RadImageNet-resnet50.pth stores the backbone as
    # nn.Sequential([conv1, bn1, relu, maxpool, layer1..4, avgpool]), so keys
    # look like "backbone.0.weight" instead of "conv1.weight". Without this
    # remap, load_state_dict(strict=False) matches ZERO keys and silently
    # leaves the model at its random init (same bug class documented in
    # src/fedmammobench/models/weight_loaders/_keymaps.py, which this mirrors).
    backbone_remap = {
        "backbone.0.": "conv1.",
        "backbone.1.": "bn1.",
        "backbone.4.": "layer1.",
        "backbone.5.": "layer2.",
        "backbone.6.": "layer3.",
        "backbone.7.": "layer4.",
    }
    remapped_state_dict = {}
    for k, v in state_dict.items():
        new_k = k
        for old_prefix, new_prefix in backbone_remap.items():
            if k.startswith(old_prefix):
                new_k = new_prefix + k[len(old_prefix):]
                break
        remapped_state_dict[new_k] = v
    state_dict = remapped_state_dict

    # Keep only the ResNet50 backbone keys
    state_dict = {
        k: v for k, v in state_dict.items()
        if k.startswith(("conv1", "bn1", "relu", "maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool"))
    }

    if len(state_dict) == 0:
        raise RuntimeError(
            "0 RadImageNet tensors matched the model after remapping — "
            "check the checkpoint's key format before continuing."
        )

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded {len(state_dict)} RadImageNet tensors into the backbone.")
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)

    # Truncate to conv1..avgpool, matching the Backbone class above, so its
    # output shape (batch, 2048, 1, 1) lines up with Classifier's Linear(2048, ...)
    encoder_layers = list(model.children())
    return nn.Sequential(*encoder_layers[:9])


backbone = load_radimagenet_backbone(radimagenet_weights_path, device=device).to(device)

Loaded 318 RadImageNet tensors into the backbone.
Missing keys: ['fc.weight', 'fc.bias']
Unexpected keys: []


In [ ]:
RUN_DIR = Path("/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/13-PregradoJulian/Federal Learning/infraestructura federada/Federal-Learning/runs/exp71_resnet50_radimagenet")
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = RUN_DIR / "best_model.pth"


def train_model(model, criterion, optimizer, num_epochs=50):
    min_valid_loss = np.inf

    epoch_bar = tqdm(range(num_epochs), desc="Epochs", unit="epoch")
    for e in epoch_bar:
        train_loss = 0.0
        model.train()
        train_bar = tqdm(train, desc=f"  train {e+1}/{num_epochs}", unit="batch", leave=False)
        for data, labels in train_bar:
            data, labels = data.to(device, dtype=torch.float), labels.to(device)
            
            # print(f"Data Max: {data.max()}, Data Min: {data.min()}, Data Mean: {data.mean()}, Data Std: {data.std()}")

            optimizer.zero_grad()
            target = model(data)
            loss = criterion(target, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        valid_loss = 0.0
        model.eval()
        val_bar = tqdm(validation, desc=f"  val   {e+1}/{num_epochs}", unit="batch", leave=False)
        with torch.no_grad():
            for data, labels in val_bar:
                data, labels = data.to(device, dtype=torch.float), labels.to(device)

                target = model(data)
                loss = criterion(target, labels)
                valid_loss += loss.item()
                val_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = train_loss / len(train)
        avg_valid_loss = valid_loss / len(validation)
        epoch_bar.set_postfix(train_loss=f"{avg_train_loss:.4f}", val_loss=f"{avg_valid_loss:.4f}")

        print(f'Epoch {e+1} \t\t Training Loss: {avg_train_loss} \t\t Validation Loss: {avg_valid_loss}')
        if min_valid_loss > valid_loss:
            print(f'Validation Loss Decreased({min_valid_loss:.6f}--->{valid_loss:.6f}) \t Saving The Model')
            min_valid_loss = valid_loss
            torch.save(model.state_dict(), BEST_MODEL_PATH)
    return model

In [ ]:
class FullModel(nn.Module):
    def __init__(self, backbone, classifier):
        super().__init__()
        self.backbone = backbone
        self.classifier = classifier

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x


model = FullModel(backbone, classifier).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [ ]:
model = train_model(model, criterion, optimizer, num_epochs=100)

Epochs:   0%|          | 0/100 [00:00<?, ?epoch/s]

  train 1/100:   0%|          | 0/117 [00:00<?, ?batch/s]

Data Max: 1.0, Data Min: -1.0, Data Mean: -0.5955116748809814, Data Std: 0.3900953531265259
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.6113190650939941, Data Std: 0.38771188259124756
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.6033533811569214, Data Std: 0.3731159269809723
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.579472005367279, Data Std: 0.4153537154197693
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.5757875442504883, Data Std: 0.4264136552810669
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.5859149694442749, Data Std: 0.3956604599952698
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.5701006054878235, Data Std: 0.4164195954799652
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.5853644013404846, Data Std: 0.4110540747642517
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.5822693109512329, Data Std: 0.40224793553352356
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.5503098368644714, Data Std: 0.4181850552558899
Data Max: 1.0, Data Min: -1.0, Data Mean: -0.5566132664680481, Data Std: 0.4104

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Evaluate the best (lowest val-loss) checkpoint on the held-out test split.
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

all_labels, all_preds, all_probs = [], [], []
with torch.no_grad():
    for data, labels in tqdm(test, desc="Test", unit="batch"):
        data = data.to(device, dtype=torch.float)
        logits = model(data)
        probs = torch.softmax(logits, dim=-1)[:, 1]  # P(class index 1)
        preds = logits.argmax(dim=-1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_auc = roc_auc_score(all_labels, all_probs)

idx_to_label = {v: k for k, v in train_processed.label_to_idx.items()}
print(f"label_to_idx: {train_processed.label_to_idx}  (positive class for AUC = {idx_to_label[1]!r})")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test ROC AUC:  {test_auc:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=[idx_to_label[0], idx_to_label[1]]))